# Diffuse Supernova Neutrino Background (DSNB)

The DSNB is the flux of neutrinos from all past core-collapse supernovae in the Universe. The class `snewpy.dsnb.DSNB` builds this flux from ordinary SNEWPY supernova models and offers the same kind of interface as they do: `get_flux` returns a `snewpy.flux.Container`, which can be integrated over time and passed to the `RateCalculator`.

For a single supernova, the time-integrated emission spectrum $dN/dE'$ is obtained from the model's public `get_transformed_spectra` method. The observed flux is then

$$\frac{d\Phi}{dE} = \frac{c}{H_0}\int_0^{z_{\rm max}} \frac{R_{\rm CC}(z)}{\sqrt{\Omega_m (1+z)^3 + \Omega_\Lambda}}\,\frac{dN}{dE'}\Big|_{E'=(1+z)E}\,dz ,$$

where $R_{\rm CC}(z)$ is the cosmic core-collapse rate. Several progenitors are combined with weights from the Salpeter initial mass function.

Reference: Li, Vagins & Wurm, arXiv:2201.12920

In [ ]:
import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np

from astropy import units as u

from snewpy.neutrino import MixingParameters
from snewpy.models.ccsn import Nakazato_2013
from snewpy.flavor_transformation import NoTransformation, AdiabaticMSW
from snewpy.rate_calculator import RateCalculator
from snewpy.dsnb import DSNB, CoreCollapseRate

mpl.rc('font', size=16)
%matplotlib inline

## Initialize Models

The DSNB takes a list of supernova models. Any model class works, because only the public interface (`get_transformed_spectra` and `time`) is used. The progenitor mass, needed for the IMF weights, is read from `model.metadata['Progenitor mass']`.

Here, the four progenitors of the `Nakazato_2013` set (revival time 100 ms, $Z=0.02$, Shen EOS) are used. If this is the first time a progenitor is used, snewpy downloads the required data files automatically.

In [ ]:
sn_models = [Nakazato_2013(progenitor_mass=m*u.solMass, revival_time=100*u.ms, metallicity=0.02, eos='shen')
             for m in (13, 20, 30, 50)]
dsnb = DSNB(sn_models)

for model, w in zip(dsnb._models, dsnb._weights):
    print(f"{model.metadata['Progenitor mass']:>10.1f}   IMF weight = {w:.3f}")

## Core-collapse rate

The redshift dependence of the supernova rate is set by `CoreCollapseRate` (Hopkins & Beacom 2006). The present-day rate $R_{\rm CC}(0)$ is the main normalisation and can be changed with `r0`.

In [ ]:
z = np.linspace(0, 5, 200)
rcc = CoreCollapseRate()

fig, ax = plt.subplots(1, figsize=(8, 5))
ax.plot(z, rcc(z), lw=2)
ax.set(xlabel='redshift $z$', ylabel=r'$R_{\rm CC}$ [yr$^{-1}$ Mpc$^{-3}$]')
ax.grid();

## DSNB flux

`get_flux(t, E, flavor_xform)` takes the same flavor transformation objects as the supernova models. The transformation describes the oscillations inside each source supernova, so the two mass orderings give different results.

The DSNB is steady-state: the flux is the same at every time in `t`, and `t` only defines the exposure window.

In [ ]:
t = np.linspace(0, 1, 2) << u.yr           # one year of exposure
E = np.linspace(1, 50, 491) << u.MeV

xforms = {'No transformation': NoTransformation(),
          'Normal ordering':   AdiabaticMSW(MixingParameters('NORMAL')),
          'Inverted ordering': AdiabaticMSW(MixingParameters('INVERTED'))}

fluxes = {name: dsnb.get_flux(t, E, xform) for name, xform in xforms.items()}

In [ ]:
from snewpy.flavor import ThreeFlavor

fig, ax = plt.subplots(1, figsize=(8, 6))
for name, flux in fluxes.items():
    phi = flux[ThreeFlavor.NU_E_BAR, 0, :].array[0, 0, :].to(1/(u.cm**2*u.s*u.MeV))
    ax.plot(E, phi, lw=2, label=name)

ax.set(xlim=(0, 50), xlabel=r'$E_\nu$ [MeV]',
       ylabel=r'$d\Phi/dE$ [cm$^{-2}$ s$^{-1}$ MeV$^{-1}$]',
       title=r'DSNB $\bar{\nu}_e$ flux')
ax.grid()
ax.legend();

## Event rates

Integrating the flux over time gives the fluence, which goes straight into the `RateCalculator`. The example below uses the 100 kt water Cherenkov detector with 30% photocoverage.

In [ ]:
rc = RateCalculator()

for name, flux in fluxes.items():
    fluence = flux.integrate('time')
    rates = rc.run(fluence, 'wc100kt30prct')
    ibd = rates['ibd'].array.sum()
    print(f'{name:18s}: {ibd:.2f} inverse beta decay events in 1 year')

## Custom weights

By default the IMF weights are computed from the progenitor masses. Explicit weights can be given instead, which also allows several models with the same progenitor mass.

```python
dsnb = DSNB(sn_models, weights=[0.5, 0.3, 0.1, 0.1])
```

A different present-day rate or cosmology can be passed with `rcc=CoreCollapseRate(r0=...)` and `cosmology=...`.